In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("set2").getOrCreate()



In [0]:
# Load enrollments data
enrollments_df = spark.read.option("header", True).option("inferSchema", True).csv("file:/Workspace/Shared/course_enrollments.csv")

# Load catalog data
catalog_df = spark.read.option("header", True).option("inferSchema", True).csv("file:/Workspace/Shared/course_catalog.csv")

enrollments_df.show()
catalog_df.show()


+--------+------+--------+-----------------+------------+----------+--------------+---------------+------+
|EnrollID|UserID|CourseID|       CourseName|    Category|EnrollDate|CompletionDate|ProgressPercent|Rating|
+--------+------+--------+-----------------+------------+----------+--------------+---------------+------+
|    E001|  U001|    C001|    Python Basics| Programming|2024-04-01|    2024-04-10|            100|     4|
|    E002|  U002|    C002|Excel for Finance|Productivity|2024-04-02|          NULL|             45|  NULL|
|    E003|  U001|    C003|  ML with PySpark|Data Science|2024-04-03|          NULL|             30|  NULL|
|    E004|  U003|    C001|    Python Basics| Programming|2024-04-04|    2024-04-20|            100|     5|
|    E005|  U004|    C004|Digital Marketing|   Marketing|2024-04-05|    2024-04-16|            100|     4|
+--------+------+--------+-----------------+------------+----------+--------------+---------------+------+

+--------+-------------+------------

# Q1. Ingestion & Time Fields

In [0]:
# Q1 - Convert EnrollDate and CompletionDate to DateType, Add DaysToComplete


from pyspark.sql.functions import to_date, datediff, when, col

df1 = enrollments_df \
    .withColumn("EnrollDate", to_date(col("EnrollDate"))) \
    .withColumn("CompletionDate", to_date(col("CompletionDate"))) \
    .withColumn("DaysToComplete", when(col("CompletionDate").isNotNull(),
                                       datediff(col("CompletionDate"), col("EnrollDate"))))

df1.show()



+--------+------+--------+-----------------+------------+----------+--------------+---------------+------+--------------+
|EnrollID|UserID|CourseID|       CourseName|    Category|EnrollDate|CompletionDate|ProgressPercent|Rating|DaysToComplete|
+--------+------+--------+-----------------+------------+----------+--------------+---------------+------+--------------+
|    E001|  U001|    C001|    Python Basics| Programming|2024-04-01|    2024-04-10|            100|     4|             9|
|    E002|  U002|    C002|Excel for Finance|Productivity|2024-04-02|          NULL|             45|  NULL|          NULL|
|    E003|  U001|    C003|  ML with PySpark|Data Science|2024-04-03|          NULL|             30|  NULL|          NULL|
|    E004|  U003|    C001|    Python Basics| Programming|2024-04-04|    2024-04-20|            100|     5|            16|
|    E005|  U004|    C004|Digital Marketing|   Marketing|2024-04-05|    2024-04-16|            100|     4|            11|
+--------+------+-------

# Q2. User Learning Path Progress

In [0]:
# Q2 - Group by UserID, course count, avg progress, and IsCompleted flag

from pyspark.sql.functions import avg, count, col, when

df2 = df1.withColumn("IsCompleted", when(col("ProgressPercent") == 100, True).otherwise(False))

df2.groupBy("UserID") \
   .agg(count("*").alias("TotalCoursesEnrolled"),
        avg("ProgressPercent").alias("AvgProgressPercent")) \
   .show()

df2.select("UserID", "ProgressPercent", "IsCompleted").show()


+------+--------------------+------------------+
|UserID|TotalCoursesEnrolled|AvgProgressPercent|
+------+--------------------+------------------+
|  U004|                   1|             100.0|
|  U002|                   1|              45.0|
|  U003|                   1|             100.0|
|  U001|                   2|              65.0|
+------+--------------------+------------------+

+------+---------------+-----------+
|UserID|ProgressPercent|IsCompleted|
+------+---------------+-----------+
|  U001|            100|       true|
|  U002|             45|      false|
|  U001|             30|      false|
|  U003|            100|       true|
|  U004|            100|       true|
+------+---------------+-----------+



# Q3. Engagement Scoring

In [0]:
# Q3 - Replace null Rating with 0, EngagementScore = ProgressPercent * Rating

from pyspark.sql.functions import coalesce

df3 = df2.withColumn("Rating", coalesce(col("Rating"), col("Rating") * 0)) \
         .withColumn("EngagementScore", col("ProgressPercent") * col("Rating"))

df3.select("EnrollID", "ProgressPercent", "Rating", "EngagementScore").show()


+--------+---------------+------+---------------+
|EnrollID|ProgressPercent|Rating|EngagementScore|
+--------+---------------+------+---------------+
|    E001|            100|     4|            400|
|    E002|             45|  NULL|           NULL|
|    E003|             30|  NULL|           NULL|
|    E004|            100|     5|            500|
|    E005|            100|     4|            400|
+--------+---------------+------+---------------+



# Q4. Identify Drop-offs

In [0]:
# Q4 - Filter records with Progress < 50 and CompletionDate is null, create view

df4 = df3.filter((col("ProgressPercent") < 50) & col("CompletionDate").isNull())

df4.createOrReplaceTempView("Dropouts")

spark.sql("SELECT * FROM Dropouts").show()


+--------+------+--------+-----------------+------------+----------+--------------+---------------+------+--------------+-----------+---------------+
|EnrollID|UserID|CourseID|       CourseName|    Category|EnrollDate|CompletionDate|ProgressPercent|Rating|DaysToComplete|IsCompleted|EngagementScore|
+--------+------+--------+-----------------+------------+----------+--------------+---------------+------+--------------+-----------+---------------+
|    E002|  U002|    C002|Excel for Finance|Productivity|2024-04-02|          NULL|             45|  NULL|          NULL|      false|           NULL|
|    E003|  U001|    C003|  ML with PySpark|Data Science|2024-04-03|          NULL|             30|  NULL|          NULL|      false|           NULL|
+--------+------+--------+-----------------+------------+----------+--------------+---------------+------+--------------+-----------+---------------+



# Q5. Joins with Metadata

In [0]:
# Q5 - Join with catalog, avg progress per instructor, and most enrolled course instructor

joined_df = df3.join(catalog_df, on="CourseID", how="inner")

# Average progress per instructor
joined_df.groupBy("Instructor").agg(avg("ProgressPercent").alias("AvgProgress")).show()

# Instructor of most enrolled course
from pyspark.sql.functions import desc

most_enrolled_course = df3.groupBy("CourseID").count().orderBy(desc("count")).first()["CourseID"]
catalog_df.filter(col("CourseID") == most_enrolled_course).select("Instructor").show()


+-------------+-----------+
|   Instructor|AvgProgress|
+-------------+-----------+
|  Zoya Sheikh|      100.0|
|   Sana Gupta|       45.0|
| Ibrahim Khan|       30.0|
|Abdullah Khan|      100.0|
+-------------+-----------+

+-------------+
|   Instructor|
+-------------+
|Abdullah Khan|
+-------------+



# Q6. Delta Lake Practice

In [0]:
# Q6 - Save as delta table, update rating to 5 for Python Basics, delete rows with 0% progress

df3.write.format("delta").mode("overwrite").save("/tmp/enrollments_delta")

from delta.tables import DeltaTable

delta_df = DeltaTable.forPath(spark, "/tmp/enrollments_delta")

# Update ratings
delta_df.update(
    condition="CourseName = 'Python Basics'",
    set={"Rating": "5"}
)

# Delete where ProgressPercent = 0
delta_df.delete("ProgressPercent = 0")

# Describe history
delta_df.history().show()


+-------+-------------------+----------------+--------------------+---------+--------------------+----+------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|          userId|            userName|operation| operationParameters| job|          notebook|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+----------------+--------------------+---------+--------------------+----+------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|      3|2025-06-19 05:34:26|7928277367239535|azuser3564_mml.lo...| OPTIMIZE|{predicate -> [],...|NULL|{2805786059712358}|0611-043414-4p180ssa|          1|SnapshotIsolation|        false|{numRemovedFiles ...|        NULL|Databricks-Runtim...|
|      2|2025-06-19 05:34:24

#  Q7. Window Functions

In [0]:
# Q7 - dense_rank for most enrolled courses, lead to find next course per user

from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, lead

course_rank_window = Window.orderBy(col("CourseID").desc())
df_ranked = df3.groupBy("CourseID").count().withColumn("Rank", dense_rank().over(course_rank_window))
df_ranked.show()

lead_window = Window.partitionBy("UserID").orderBy("EnrollDate")
df_lead = df3.withColumn("NextCourse", lead("CourseName").over(lead_window))
df_lead.select("UserID", "CourseName", "EnrollDate", "NextCourse").show()


+--------+-----+----+
|CourseID|count|Rank|
+--------+-----+----+
|    C004|    1|   1|
|    C003|    1|   2|
|    C002|    1|   3|
|    C001|    2|   4|
+--------+-----+----+

+------+-----------------+----------+---------------+
|UserID|       CourseName|EnrollDate|     NextCourse|
+------+-----------------+----------+---------------+
|  U001|    Python Basics|2024-04-01|ML with PySpark|
|  U001|  ML with PySpark|2024-04-03|           NULL|
|  U002|Excel for Finance|2024-04-02|           NULL|
|  U003|    Python Basics|2024-04-04|           NULL|
|  U004|Digital Marketing|2024-04-05|           NULL|
+------+-----------------+----------+---------------+



# Q8. SQL Logic for Dashboard Views

In [0]:
# Q8 - Create views: daily_enrollments, category_performance, top_3_courses

df3.createOrReplaceTempView("enrollments")

# daily_enrollments
spark.sql("""
    CREATE OR REPLACE TEMP VIEW daily_enrollments AS
    SELECT EnrollDate, COUNT(*) as TotalEnrollments
    FROM enrollments
    GROUP BY EnrollDate
""")

# category_performance
spark.sql("""
    CREATE OR REPLACE TEMP VIEW category_performance AS
    SELECT Category, AVG(Rating) as AvgRating
    FROM enrollments
    GROUP BY Category
""")

# top_3_courses
spark.sql("""
    CREATE OR REPLACE TEMP VIEW top_3_courses AS
    SELECT CourseName, COUNT(*) as Enrollments
    FROM enrollments
    GROUP BY CourseName
    ORDER BY Enrollments DESC
    LIMIT 3
""")

spark.sql("SELECT * FROM daily_enrollments").show()
spark.sql("SELECT * FROM category_performance").show()
spark.sql("SELECT * FROM top_3_courses").show()


+----------+----------------+
|EnrollDate|TotalEnrollments|
+----------+----------------+
|2024-04-02|               1|
|2024-04-01|               1|
|2024-04-04|               1|
|2024-04-05|               1|
|2024-04-03|               1|
+----------+----------------+

+------------+---------+
|    Category|AvgRating|
+------------+---------+
| Programming|      4.5|
|Productivity|     NULL|
|   Marketing|      4.0|
|Data Science|     NULL|
+------------+---------+

+-----------------+-----------+
|       CourseName|Enrollments|
+-----------------+-----------+
|    Python Basics|          2|
|Digital Marketing|          1|
|Excel for Finance|          1|
+-----------------+-----------+



# Q9. Time Travel

In [0]:
# Q9 - View version before update/delete using VERSION AS OF

df_version1 = spark.read.format("delta").option("versionAsOf", 0).load("/tmp/enrollments_delta")
df_version1.show()


+--------+------+--------+-----------------+------------+----------+--------------+---------------+------+--------------+-----------+---------------+
|EnrollID|UserID|CourseID|       CourseName|    Category|EnrollDate|CompletionDate|ProgressPercent|Rating|DaysToComplete|IsCompleted|EngagementScore|
+--------+------+--------+-----------------+------------+----------+--------------+---------------+------+--------------+-----------+---------------+
|    E001|  U001|    C001|    Python Basics| Programming|2024-04-01|    2024-04-10|            100|     4|             9|       true|            400|
|    E002|  U002|    C002|Excel for Finance|Productivity|2024-04-02|          NULL|             45|  NULL|          NULL|      false|           NULL|
|    E003|  U001|    C003|  ML with PySpark|Data Science|2024-04-03|          NULL|             30|  NULL|          NULL|      false|           NULL|
|    E004|  U003|    C001|    Python Basics| Programming|2024-04-04|    2024-04-20|            100| 

# Q10. Export Reporting

In [0]:
# Q10 - Write to JSON partitioned by Category and Save summary to Parquet

# Write to JSON
df3.write.mode("overwrite").partitionBy("Category").json("/tmp/json_output")

# Create summary DF
summary_df = df3.groupBy("CourseName") \
                .agg(count("*").alias("TotalEnrollments"),
                     avg("Rating").alias("AvgRating"),
                     avg("ProgressPercent").alias("AvgProgress"))

# Save to Parquet
summary_df.write.mode("overwrite").parquet("/tmp/summary_parquet")
